# ResNet50 sobre el dataset regional

Entrenamiento aparte del MobileNetV2 regional. Usa las mismas fotos y el mismo reparto de `regional_colombia/dataset_29`: **10201 train / 2178 val / 2201 test**, 29 especies, semilla 42.

La receta es la del MobileNetV2 de 29 clases: entrada 224×224, la misma augmentación, Dropout 0.4, dos fases (cabeza con la base congelada y luego las últimas 40 capas) y parada temprana. No hay multa L1/L2. El preprocesamiento es el de ResNet50 (BGR y resta de la media de ImageNet).

No reemplaza `modelo_regional_29clases.h5`. Los archivos salen con el sufijo `resnet50_regional_29`.

Ejecutar con el kernel del entorno `.venv` (TensorFlow 2.10 + DirectML), desde la raíz del repositorio.


In [1]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
os.environ["DML_VISIBLE_DEVICES"] = "0"
os.environ["MPLBACKEND"] = "Agg"

from pathlib import Path
import json
import time

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix

RAIZ = Path.cwd()
if not (RAIZ / "regional_colombia" / "dataset_29").is_dir():
    RAIZ = Path(r"D:\juandmm1233\NoElectivaIII")

DATASET = RAIZ / "regional_colombia" / "dataset_29"
SALIDA = RAIZ / "regional_colombia" / "resultados"
SALIDA.mkdir(parents=True, exist_ok=True)

IMG = (224, 224)
BATCH = 16
SEMILLA = 42
EPOCHS_FASE1 = 25
EPOCHS_FASE2 = 20
ESPERADO = {"train": 10201, "val": 2178, "test": 2201}
PREFIJO = "resnet50_regional_29"

tf.keras.utils.set_random_seed(SEMILLA)
gpus = tf.config.list_physical_devices("GPU")
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass

print("TensorFlow", tf.__version__)
print("GPU:", gpus or "no detectada")
print("Dataset:", DATASET)


TensorFlow 2.10.1
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Dataset: D:\juandmm1233\NoElectivaIII\regional_colombia\dataset_29


In [2]:
def contar(split):
    carpeta = DATASET / split
    if not carpeta.is_dir():
        raise FileNotFoundError(f"Falta {carpeta}")
    return sum(1 for ruta in carpeta.rglob("*") if ruta.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"})

conteo = {split: contar(split) for split in ESPERADO}
print(conteo)
if conteo != ESPERADO:
    raise RuntimeError(f"El reparto no es el del MobileNetV2 regional. Esperado {ESPERADO}, hay {conteo}.")

def cargar(split, shuffle):
    return tf.keras.utils.image_dataset_from_directory(
        DATASET / split,
        image_size=IMG,
        batch_size=BATCH,
        shuffle=shuffle,
        seed=SEMILLA,
        label_mode="int",
    )

train_ds = cargar("train", True)
val_ds = cargar("val", False)
test_ds = cargar("test", False)
clases = list(train_ds.class_names)
if clases != list(val_ds.class_names) or clases != list(test_ds.class_names):
    raise RuntimeError("Las clases de train, val y test no coinciden.")
print(f"Clases: {len(clases)}")

autotune = tf.data.AUTOTUNE
cache = SALIDA / "cache_resnet50_29"
cache.mkdir(parents=True, exist_ok=True)
train_ds = train_ds.cache(str(cache / f"train_b{BATCH}")).shuffle(800, seed=SEMILLA).prefetch(autotune)
val_ds = val_ds.cache(str(cache / f"val_b{BATCH}")).prefetch(autotune)
test_ds = test_ds.cache(str(cache / f"test_b{BATCH}")).prefetch(autotune)


{'train': 10201, 'val': 2178, 'test': 2201}


Found 10201 files belonging to 29 classes.


Found 2178 files belonging to 29 classes.
Found 2201 files belonging to 29 classes.
Clases: 29


In [3]:
def hacer_augmentacion():
    return tf.keras.Sequential([
        tf.keras.layers.RandomFlip("horizontal_and_vertical"),
        tf.keras.layers.RandomRotation(0.25),
        tf.keras.layers.RandomZoom(0.25),
        tf.keras.layers.RandomTranslation(0.15, 0.15),
        tf.keras.layers.RandomContrast(0.3),
        tf.keras.layers.RandomBrightness(0.25, value_range=(0, 255)),
    ], name="augmentacion")

def construir(num_clases):
    base = tf.keras.applications.ResNet50(
        input_shape=IMG + (3,),
        include_top=False,
        weights="imagenet",
    )
    base.trainable = False
    entradas = tf.keras.Input(shape=IMG + (3,))
    x = hacer_augmentacion()(entradas)
    x = tf.keras.applications.resnet50.preprocess_input(x)
    x = base(x, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.4)(x)
    salidas = tf.keras.layers.Dense(num_clases, activation="softmax", name="cabeza")(x)
    modelo = tf.keras.Model(entradas, salidas, name="resnet50_regional_29")
    return modelo, base

def callbacks():
    return [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_accuracy",
            patience=6,
            restore_best_weights=True,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=3,
        ),
    ]

modelo, base = construir(len(clases))
modelo.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
print("Fase 1: ResNet50 congelado, aprende la cabeza")
modelo.summary()
t0 = time.perf_counter()
hist1 = modelo.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_FASE1,
    callbacks=callbacks(),
)

base.trainable = True
for capa in base.layers[:-40]:
    capa.trainable = False
modelo.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
print("Fase 2: ultimas 40 capas de ResNet50")
hist2 = modelo.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_FASE2,
    callbacks=callbacks(),
)
minutos = (time.perf_counter() - t0) / 60
print(f"Entrenamiento: {minutos:.1f} min")


Fase 1: ResNet50 congelado, aprende la cabeza
Model: "resnet50_regional_29"


_________________________________________________________________


 Layer (type)                Output Shape              Param #   


 input_2 (InputLayer)        [(None, 224, 224, 3)]     0         


 augmentacion (Sequential)   (None, 224, 224, 3)       0         


 tf.__operators__.getitem (S  (None, 224, 224, 3)      0         


... (28,808 outputs left out)

637/638 [============================>.] - ETA: 0s - loss: 0.3966 - accuracy: 0.8756

638/638 [==============================] - ETA: 0s - loss: 0.3965 - accuracy: 0.8756

638/638 [==============================] - 225s 344ms/step - loss: 0.3965 - accuracy: 0.8756 - val_loss: 0.9846 - val_accuracy: 0.7787 - lr: 1.2500e-06


Entrenamiento: 164.3 min


In [4]:
def unir(hist1, hist2):
    unidos = {}
    for clave in ("accuracy", "val_accuracy", "loss", "val_loss"):
        unidos[clave] = hist1.history.get(clave, []) + hist2.history.get(clave, [])
    return unidos, len(hist1.history.get("accuracy", []))

historial, corte = unir(hist1, hist2)
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(historial["accuracy"], label="train")
ax[0].plot(historial["val_accuracy"], label="val")
ax[0].axvline(corte - 0.5, color="gray", linestyle="--", label="fine-tuning")
ax[0].set_title("Accuracy")
ax[0].set_xlabel("Epoca")
ax[0].legend()
ax[1].plot(historial["loss"], label="train")
ax[1].plot(historial["val_loss"], label="val")
ax[1].axvline(corte - 0.5, color="gray", linestyle="--", label="fine-tuning")
ax[1].set_title("Loss")
ax[1].set_xlabel("Epoca")
ax[1].legend()
fig.suptitle("ResNet50 regional")
fig.tight_layout()
fig.savefig(SALIDA / f"curvas_{PREFIJO}.png", dpi=120)
plt.show()

ruta_modelo = SALIDA / f"modelo_{PREFIJO}.h5"
modelo.save(ruta_modelo, include_optimizer=False)

perdida, accuracy = modelo.evaluate(test_ds, verbose=1)
y_true = np.concatenate([y.numpy() for _, y in test_ds])
y_prob = modelo.predict(test_ds, verbose=1)
y_pred = y_prob.argmax(axis=1)
top3 = float(np.mean(tf.keras.metrics.sparse_top_k_categorical_accuracy(y_true, y_prob, k=3).numpy()))
reporte = classification_report(y_true, y_pred, target_names=clases, digits=3, output_dict=True)
texto = classification_report(y_true, y_pred, target_names=clases, digits=3)
print(texto)
(SALIDA / f"reporte_{PREFIJO}.txt").write_text(texto, encoding="utf-8")

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(18, 16))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Greens",
    xticklabels=clases, yticklabels=clases, cbar=True, annot_kws={"size": 6},
)
plt.xticks(fontsize=7, rotation=90)
plt.yticks(fontsize=7)
plt.xlabel("Predicho")
plt.ylabel("Real")
plt.title("Matriz de confusion - ResNet50 regional, 29 especies")
plt.tight_layout()
plt.savefig(SALIDA / f"matriz_{PREFIJO}.png", dpi=120)
plt.show()

metricas = {
    "modelo": "ResNet50 regional 29",
    "dataset": "iNaturalist Colombia/Tolima + fotos de huerta del equipo (dataset_29)",
    "comparado_con": "MobileNetV2_regional_29",
    "loss_test": float(perdida),
    "accuracy_test": float(accuracy),
    "top3_accuracy_test": top3,
    "f1_ponderado": round(float(reporte["weighted avg"]["f1-score"]), 3),
    "f1_por_clase": {clase: round(float(reporte[clase]["f1-score"]), 3) for clase in clases},
    "imagenes_test": int(len(y_true)),
    "errores_test": int(np.sum(y_true != y_pred)),
    "parametros": int(modelo.count_params()),
    "minutos_entrenamiento": round(minutos, 1),
    "tamano_mb": round(ruta_modelo.stat().st_size / (1024 * 1024), 2),
    "batch": BATCH,
    "epochs_fase1_pedidas": EPOCHS_FASE1,
    "epochs_fase2_pedidas": EPOCHS_FASE2,
    "epocas_fase1_corridas": len(hist1.history.get("accuracy", [])),
    "epocas_fase2_corridas": len(hist2.history.get("accuracy", [])),
    "mejor_val_accuracy": float(max(historial["val_accuracy"])) if historial["val_accuracy"] else None,
    "imagen": 224,
    "split": "70/15/15",
    "train": ESPERADO["train"],
    "val": ESPERADO["val"],
    "test": ESPERADO["test"],
    "semilla": SEMILLA,
    "clases": len(clases),
    "regularizacion": "Dropout 0.4, augmentacion y parada temprana. Sin L1/L2.",
}
(SALIDA / f"metricas_{PREFIJO}.json").write_text(
    json.dumps(metricas, ensure_ascii=False, indent=2), encoding="utf-8"
)
print(f"Accuracy test: {metricas['accuracy_test']:.3f}")
print(f"Top-3 test:    {metricas['top3_accuracy_test']:.3f}")
print(f"F1 ponderado:  {metricas['f1_ponderado']:.3f}")
print(f"Modelo:        {ruta_modelo}")
metricas


C:\Users\DinamizaTI\AppData\Local\Temp\ipykernel_32608\3874534705.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


  1/138 [..............................] - ETA: 16s - loss: 0.8806 - accuracy: 0.7500

  3/138 [..............................] - ETA: 5s - loss: 0.6811 - accuracy: 0.8125 

  5/138 [>.............................] - ETA: 5s - loss: 0.6255 - accuracy: 0.8375

  7/138 [>.............................] - ETA: 5s - loss: 0.9639 - accuracy: 0.7589

  9/138 [>.............................] - ETA: 5s - loss: 0.9116 - accuracy: 0.7639

... (141 outputs left out)

                                      precision    recall  f1-score   support

                Aloe_vera__aloe_vera      0.823     0.844     0.833        77
         Baccharis_latifolia__chilca      0.754     0.676     0.713        68
              Bixa_orellana__achiote      0.898     0.859     0.878        92
       Bocconia_frutescens__trompeto      0.622     0.824     0.709        68
    Calendula_officinalis__calendula      0.880     0.869     0.874        84
          Cattleya_trianae__cattleya      0.941     0.941     0.941        68
            Cecropia_peltata__yarumo      0.536     0.544     0.540        68
Ceroxylon_quindiuense__palma_de_cera      0.913     0.926     0.920        68
       Dysphania_ambrosioides__paico      0.647     0.677     0.662        65
Equisetum_bogotense__cola_de_caballo      0.932     0.809     0.866        68
Eryngium_foetidum__cilantro_cimarron      0.727     0.824     0.772        68
    Espeletia_grandiflora__frailejon      0.968     0.897     0

Accuracy test: 0.765
Top-3 test:    0.892
F1 ponderado:  0.768
Modelo:        D:\juandmm1233\NoElectivaIII\regional_colombia\resultados\modelo_resnet50_regional_29.h5


C:\Users\DinamizaTI\AppData\Local\Temp\ipykernel_32608\3874534705.py:52: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


{'modelo': 'ResNet50 regional 29',
 'dataset': 'iNaturalist Colombia/Tolima + fotos de huerta del equipo (dataset_29)',
 'comparado_con': 'MobileNetV2_regional_29',
 'loss_test': 1.0394034385681152,
 'accuracy_test': 0.7651067972183228,
 'top3_accuracy_test': 0.8923216462135315,
 'f1_ponderado': 0.768,
 'f1_por_clase': {'Aloe_vera__aloe_vera': 0.833,
  'Baccharis_latifolia__chilca': 0.713,
  'Bixa_orellana__achiote': 0.878,
  'Bocconia_frutescens__trompeto': 0.709,
  'Calendula_officinalis__calendula': 0.874,
  'Cattleya_trianae__cattleya': 0.941,
  'Cecropia_peltata__yarumo': 0.54,
  'Ceroxylon_quindiuense__palma_de_cera': 0.92,
  'Dysphania_ambrosioides__paico': 0.662,
  'Equisetum_bogotense__cola_de_caballo': 0.866,
  'Eryngium_foetidum__cilantro_cimarron': 0.772,
  'Espeletia_grandiflora__frailejon': 0.931,
  'Gliricidia_sepium__matarraton': 0.484,
  'Lippia_alba__pronto_alivio': 0.843,
  'Melissa_officinalis__toronjil': 0.8,
  'Moringa_oleifera__moringa': 0.607,
  'Petiveria_allia